# Basics (Clean Draft)

This notebook is a simplified first draft using the external `Party` class from `src/party.py`.

Goals covered:
- Confidentiality (symmetric and asymmetric encryption)
- Authentication (nonce challenge)
- Message Integrity (hash + encrypted hash)
- Everything Together (session key exchange + protected message transfer)

Queue-based transmission is used so a MITM step can be inserted later.

In [15]:
import hashlib
from src.party import Party

In [16]:
partyA = Party("Party A")
partyB = Party("Party B")
queue_a_to_b = []
queue_b_to_a = []

print(f"[Setup] Created parties: {partyA.name}, {partyB.name}")
print("[Setup] Created two queues: A->B and B->A")

[Setup] Created parties: Party A, Party B
[Setup] Created two queues: A->B and B->A


## 1. Confidentiality

### Symmetric Encryption (Cryptography)
Encrypt a message with a session key and decrypt it again.

In [17]:
message = "Hello from symmetric encryption"
session_key = partyA.generate_session_key()
ciphertext = partyA.apply_encryption(message, lambda m: partyA.encrypt_with_session_key(m))
decrypted = partyA.apply_decryption(ciphertext, lambda c: partyA.decrypt_with_session_key(c))
success = decrypted == message

print("[Confidentiality-Symmetric] Session key generated.")
print(f"[Confidentiality-Symmetric] Plaintext: {message}")
print(f"[Confidentiality-Symmetric] Ciphertext: {ciphertext[:80]}...")
print(f"[Confidentiality-Symmetric] Decrypted: {decrypted}")
print(f"[Confidentiality-Symmetric] Success: {success}")

[Confidentiality-Symmetric] Session key generated.
[Confidentiality-Symmetric] Plaintext: Hello from symmetric encryption
[Confidentiality-Symmetric] Ciphertext: gAAAAABpwZS2tc_7PuPcPqEfGRseH6ngc7hQHhanldALm8qxe5oYrpefXaqayX4juCH_xETnwwTpWlbY...
[Confidentiality-Symmetric] Decrypted: Hello from symmetric encryption
[Confidentiality-Symmetric] Success: True


### Asymmetric Encryption (Cryptography)
Encrypt with Party B's public key and decrypt with Party B's private key.

In [18]:
message = "Hello from asymmetric encryption"
ciphertext = partyA.apply_encryption(
    message,
    lambda m: partyA.encrypt_with_public_key(partyB.public_key, m)
)
decrypted = partyB.apply_decryption(ciphertext, partyB.decrypt_with_private_key)
success = decrypted == message

print("[Confidentiality-Asymmetric] Encrypted with B public key.")
print(f"[Confidentiality-Asymmetric] Plaintext: {message}")
print(f"[Confidentiality-Asymmetric] Ciphertext: {ciphertext[:80]}...")
print(f"[Confidentiality-Asymmetric] Decrypted: {decrypted}")
print(f"[Confidentiality-Asymmetric] Success: {success}")

[Confidentiality-Asymmetric] Encrypted with B public key.
[Confidentiality-Asymmetric] Plaintext: Hello from asymmetric encryption
[Confidentiality-Asymmetric] Ciphertext: qK47fCJA7t+sjcHFYma0DSoq6PgCT3HggEGje3l01ul46QA1KDWp6u/XvxCHT5A+YnXO0+/xJZkuojSt...
[Confidentiality-Asymmetric] Decrypted: Hello from asymmetric encryption
[Confidentiality-Asymmetric] Success: True


## 2. Authentication (Nonce)

Flow:
1. Party B generates a nonce and sends it to Party A.
2. Party A encrypts nonce with Party A private key.
3. Party B decrypts with Party A public key and compares.

In [19]:
queue_a_to_b.clear()
queue_b_to_a.clear()

nonce = partyB.generate_nonce()
partyB.push_message_to_queue(str(nonce), queue_b_to_a)
received_nonce = partyA.pop_message_from_queue(queue_b_to_a)
encrypted_nonce = partyA.apply_encryption(received_nonce, partyA.encrypt_with_private_key)
partyA.push_message_to_queue(encrypted_nonce, queue_a_to_b)
received_encrypted_nonce = partyB.pop_message_from_queue(queue_a_to_b)
decrypted_nonce = partyB.apply_decryption(ö
    received_encrypted_nonce,
    lambda c: partyB.decrypt_with_public_key(c, partyA.public_key)
)
authenticated = decrypted_nonce == str(nonce)

print(f"[Authentication] Nonce generated by B: {nonce}")
print("[Authentication] B -> A: nonce sent.")
print("[Authentication] A -> B: encrypted nonce sent.")
print(f"[Authentication] Nonce decrypted by B: {decrypted_nonce}")
print(f"[Authentication] Success: {authenticated}")

[Authentication] Nonce generated by B: 284700462
[Authentication] B -> A: nonce sent.
[Authentication] A -> B: encrypted nonce sent.
[Authentication] Nonce decrypted by B: 284700462
[Authentication] Success: True


## 3. Integrity (Encrypted Hash + Cleartext Message)

Flow:
1. Party A hashes the message.
2. Party A encrypts hash with Party A private key.
3. Party A combines encrypted_hash + message and sends package.
4. Party B splits package, decrypts hash with Party A public key, re-hashes message, compares hashes.

In [20]:
queue_a_to_b.clear()

message = "Transfer CHF 100 to account 12345"
hash_a = partyA.apply_hashing(message, hashlib.sha256)
encrypted_hash = partyA.apply_encryption(hash_a, partyA.encrypt_with_private_key)
combined_package = partyA.combine_hash_and_message(message, encrypted_hash)
partyA.push_message_to_queue(combined_package, queue_a_to_b)

received_package = partyB.pop_message_from_queue(queue_a_to_b)
received_encrypted_hash, received_message = partyB.split_hash_and_message(
    received_package,
    len(encrypted_hash)
)
decrypted_hash = partyB.apply_decryption(
    received_encrypted_hash,
    lambda c: partyB.decrypt_with_public_key(c, partyA.public_key)
)
hash_b = partyB.apply_hashing(received_message, hashlib.sha256)
integrity_ok = decrypted_hash == hash_b

print(f"[Integrity] Message sent in clear: {received_message}")
print(f"[Integrity] Hash from A (after decrypt): {decrypted_hash}")
print(f"[Integrity] Hash at B: {hash_b}")
print(f"[Integrity] Success: {integrity_ok}")

[Integrity] Message sent in clear: Transfer CHF 100 to account 12345
[Integrity] Hash from A (after decrypt): 98038b676e8f6a2b6333d63aa0968b10c251fe0fa375dcee3d61c5c49c08ce20
[Integrity] Hash at B: 98038b676e8f6a2b6333d63aa0968b10c251fe0fa375dcee3d61c5c49c08ce20
[Integrity] Success: True


## 4. Everything Together

Step 1: Exchange session key using asymmetric encryption.

Step 2: Send protected package:
- Party A computes hash(message)
- Party A encrypts hash with private key
- Party A combines encrypted_hash + message
- Party A encrypts combined package with session key
- Party B decrypts package with session key
- Party B splits package, decrypts hash with Party A public key, and verifies integrity

In [21]:
queue_a_to_b.clear()

# Step 1: Exchange session key
session_key_a = partyA.generate_session_key()
session_key_text = session_key_a.decode("utf-8")
encrypted_session_key = partyA.encrypt_with_public_key(partyB.public_key, session_key_text)
partyA.push_message_to_queue(encrypted_session_key, queue_a_to_b)
received_encrypted_session_key = partyB.pop_message_from_queue(queue_a_to_b)
session_key_b_text = partyB.decrypt_with_private_key(received_encrypted_session_key)
partyB.session_key = session_key_b_text.encode("utf-8")

# Step 2: Send message with encrypted hash inside session encryption
message = "Meet at 10:00 near the main station"
hash_a = partyA.apply_hashing(message, hashlib.sha256)
encrypted_hash_a = partyA.encrypt_with_private_key(hash_a)
combined = partyA.combine_hash_and_message(message, encrypted_hash_a)
encrypted_combined = partyA.encrypt_with_session_key(combined)
partyA.push_message_to_queue(encrypted_combined, queue_a_to_b)
received_encrypted_combined = partyB.pop_message_from_queue(queue_a_to_b)
received_combined = partyB.decrypt_with_session_key(received_encrypted_combined)
received_encrypted_hash, received_message = partyB.split_hash_and_message(
    received_combined,
    len(encrypted_hash_a)
)
received_hash_from_a = partyB.decrypt_with_public_key(received_encrypted_hash, partyA.public_key)
local_hash_b = partyB.apply_hashing(received_message, hashlib.sha256)
transfer_ok = received_hash_from_a == local_hash_b

print(f"[All] Session key exchange success: {partyA.session_key == partyB.session_key}")
print(f"[All] Received message: {received_message}")
print(f"[All] Hash from A: {received_hash_from_a}")
print(f"[All] Local hash at B: {local_hash_b}")
print(f"[All] Success: {transfer_ok}")

[All] Session key exchange success: True
[All] Received message: Meet at 10:00 near the main station
[All] Hash from A: 85bba0db4c93870f86a58867da7c1844e8fe0e025db7f6035610dbd9f25bec93
[All] Local hash at B: 85bba0db4c93870f86a58867da7c1844e8fe0e025db7f6035610dbd9f25bec93
[All] Success: True
